# 04｜O2O 加算收益与持有段计数分析

本 Notebook 的输入是 **03 Notebook 已生成的含持有期精简八列表**，不重新生成八列。

- 每行是实际执行日：前一实际交易日收盘形成，当前执行日开盘可执行；
- 收益使用执行日开盘到下一实际交易日开盘的 O2O；
- 曲线使用 `1 + cumsum(持仓 × O2O)` 的加算口径，不做复利；
- 最新行如果还没有下一实际交易日开盘价，保留信号和审计行，但不进入收益评价，不填 0；
- 本 Notebook 负责收益、净值、风险和持有段/段计数；逐年拆解及 2026 年原因分析由 05 完成。

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

PACKAGE_ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / 'src' / 'generate_compact_output.py').is_file()
)
SRC_ROOT = PACKAGE_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

SPOT_TEXT = os.environ.get(
    'COMPANY_SPOT_PATH',
    '/home/hzy/cta/IC数据更新_最终固化版/现货最终版/CSI500_SPOT_md_eod_raw_最终版.parquet',
).strip()
if not SPOT_TEXT or not Path(SPOT_TEXT).expanduser().is_absolute():
    raise RuntimeError('请设置 COMPANY_SPOT_PATH 为本地米筐现货的绝对路径。')
SPOT_PATH = Path(SPOT_TEXT).expanduser().resolve()
HOLDING_PATH = Path(os.environ.get(
    'HOLDING_EIGHT_PATH',
    str(PACKAGE_ROOT / 'runtime_outputs_holding_period' / '含持有期八列表.csv'),
)).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get(
    'ANALYSIS_04_OUTPUT_DIR',
    str(PACKAGE_ROOT / 'runtime_outputs_04_returns'),
)).expanduser().resolve()
if not HOLDING_PATH.is_absolute() or not OUTPUT_DIR.is_absolute():
    raise RuntimeError('04 的八列表路径和输出目录都必须是绝对路径。')

from reproduce_remote_o2o import run_stage_04

print('冻结包目录：', PACKAGE_ROOT)
print('本地现货：', SPOT_PATH)
print('03 八列表：', HOLDING_PATH)
print('04 输出：', OUTPUT_DIR)

## 1. 运行 O2O 加算收益与持有段分析

In [ ]:
metadata = run_stage_04(SPOT_PATH, HOLDING_PATH, OUTPUT_DIR)
risk = pd.read_csv(OUTPUT_DIR / 'O2O加算风险指标.csv', encoding='utf-8-sig')
segment_counts = pd.read_csv(OUTPUT_DIR / '持有段计数_按系列.csv', encoding='utf-8-sig')
daily = pd.read_csv(OUTPUT_DIR / 'O2O加算逐日收益与状态.csv', encoding='utf-8-sig', parse_dates=['实际执行日', '推定形成日'])
display(risk)
display(segment_counts)
display(daily.tail(10))
print('生成文件数：', len(metadata['generated_files']))

## 2. 日期、收益和最新行检查

这里明确检查：实际执行日行的收益只取当前执行日开盘到下一实际交易日开盘；最新行没有下一开盘时只能是未评价状态，不能被补成占位收益。

In [ ]:
if not daily['形成日早于执行日'].all():
    raise AssertionError('发现形成日不早于执行日的行')
valid = daily['O2O可评价']
expected_o2o = daily.loc[valid, '下一交易日开盘'] / daily.loc[valid, '执行日开盘'] - 1.0
actual_o2o = daily.loc[valid, '执行日O2O']
if (expected_o2o - actual_o2o).abs().max() > 1e-12:
    raise AssertionError('O2O 不是执行日开盘到下一实际交易日开盘')
if daily['实际执行日'].duplicated().any():
    raise AssertionError('实际执行日重复')
latest = daily.iloc[-1]
print('最新形成日：', latest['推定形成日'])
print('最新执行日：', latest['实际执行日'])
print('最新行有下一开盘价：', bool(latest['O2O可评价']))
print('可评价行数：', int(valid.sum()), '/', len(daily))
print('检查通过：没有用占位值补齐最新收益。')

## 3. 输出位置

04 会输出 `O2O加算逐日收益与状态.csv`、`持有段明细.csv`、`持有段计数_按系列.csv`、风险指标和四张收益曲线。05 Notebook 只读取这些 04 结果做逐年与 2026 分析。

另外，04 会单独输出只加入 `0→-1`、`0→+1` 的含持有期场景：`持有段明细_仅零段反转.csv`（每一段的开始执行日、结束执行日、持有交易日数、来源）和 `持有段天数分布_仅零段反转.csv`。该场景不加入 `-1→0`、`+1→0`。

## 4. 只加入两个零段反转后的持仓段日期

下面的场景从基础三状态出发，只在基础状态为 0 时叠加 03 生成的连续 `0转-1`、`0转+1` 持有路径；`-1反转` 和 `+1反转` 完全不参与。`start_date` 和 `end_date` 都是实际执行日，`holding_days` 是连续交易日数量。

In [ ]:
zero_segments = pd.read_csv(OUTPUT_DIR / '持有段明细_仅零段反转.csv', encoding='utf-8-sig', parse_dates=['start_date', 'end_date'])
zero_duration = pd.read_csv(OUTPUT_DIR / '持有段天数分布_仅零段反转.csv', encoding='utf-8-sig')
print('场景：基础三状态 + 0→-1 / 0→+1 持有路径；不加入 -1→0 / +1→0')
display(zero_duration.pivot(index='holding_days', columns='state_label', values='segments').fillna(0).astype(int))
display(zero_segments[['run_id', 'state', 'start_date', 'end_date', 'holding_days', 'source_detail', 'zero_transfer_days', 'segment_return', 'return_available']])
print('以上明细覆盖每一个连续段；日期均为实际执行日，不是形成日。')

## 5. 四个反转全部加入后的碎片诊断

这里使用最终 `Adj 1545`：四个反转全部参与。重点查看三状态的完整持有期分布，以及一日/两日段的日期、形成日和形成机制。该诊断不自动把短段合并，因为合并会改变冻结状态定义。

In [ ]:
final_duration = pd.read_csv(OUTPUT_DIR / '最终三状态_持有段天数分布.csv', encoding='utf-8-sig')
short_detail = pd.read_csv(OUTPUT_DIR / '最终三状态_一二日段明细.csv', encoding='utf-8-sig', parse_dates=['start_date', 'end_date', 'start_formation_date', 'end_formation_date'])
mechanism = pd.read_csv(OUTPUT_DIR / '最终三状态_一二日段形成机制统计.csv', encoding='utf-8-sig')
print('最终四反转 Adj 1545 的持有天数分布：')
display(final_duration.pivot(index='holding_days', columns='state_label', values='segments').fillna(0).astype(int))
print('一日/两日段形成机制：')
display(mechanism)
print('一日/两日段日期明细：')
display(short_detail)